<a href="https://colab.research.google.com/github/aurangozebsajib/All-tools-factory/blob/main/Hugging_Face_Use_Via_API_auto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install huggingface_hub requests beautifulsoup4 pandas openpyxl tqdm

import time, requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm import tqdm
from huggingface_hub import list_spaces

CATEGORY = "video-generation"
LIMIT = 500 # change to 2000 if you want all
DELAY = 1.2

HEADERS = {"User-Agent": "Mozilla/5.0"}

def get_api_page_content(space_id):
    result = {"has_api_button": False, "api_docs_url": "", "api_docs_content": "", "error": ""}
    space_url = f"https://huggingface.co/spaces/{space_id}"
    try:
        resp = requests.get(space_url, headers=HEADERS, timeout=15)
        text = resp.text
        if "Use via API" in text or "gradio_api" in text.lower():
            result["has_api_button"] = True

        # Try to get real API docs
        subdomain = space_id.replace("/", "-").replace("_", "-").lower()
        urls = [
            f"https://{subdomain}.hf.space/gradio_api/openapi",
            f"https://{subdomain}.hf.space/api",
            f"https://huggingface.co/spaces/{space_id}/api",
        ]
        for u in urls:
            try:
                r = requests.get(u, headers=HEADERS, timeout=10)
                if r.status_code == 200 and len(r.text) > 200:
                    result["api_docs_url"] = u
                    result["api_docs_content"] = r.text[:30000]
                    break
            except: continue
        if result["has_api_button"] and not result["api_docs_content"]:
            result["api_docs_url"] = space_url
            result["api_docs_content"] = text[:30000]
    except Exception as e:
        result["error"] = str(e)
    return result

print("Fetching list...")
spaces = list(list_spaces(filter=CATEGORY, limit=LIMIT, sort="likes"))
print(f"Found {len(spaces)} spaces")

rows = []
for s in tqdm(spaces):
    space_id = s.id
    api = get_api_page_content(space_id)
    rows.append({
        "name": space_id.split("/")[-1],
        "url": f"https://huggingface.co/spaces/{space_id}",
        "likes": getattr(s, 'likes', 0),
        "has_use_via_api_button": api["has_api_button"],
        "api_docs_url": api["api_docs_url"],
        "api_docs_content": api["api_docs_content"],
    })
    time.sleep(DELAY)

df = pd.DataFrame(rows)
df.to_csv("hf_video_generation_spaces.csv", index=False)
df.to_excel("hf_video_generation_spaces.xlsx", index=False)
print("DONE")
df.head(10)

Fetching list...
Found 354 spaces


100%|██████████| 354/354 [20:06<00:00,  3.41s/it]


DONE


,name,url,likes,has_use_via_api_button,api_docs_url,api_docs_content
0,CogVideoX-5B-Space,https://huggingface.co/spaces/zai-org/CogVideo...,1049,False,https://zai-org-cogvideox-5b-space.hf.space/api,"<!doctype html>\n<html\n\tlang=""en""\n\tstyle=""..."
1,CogVideoX-2B-Space,https://huggingface.co/spaces/zai-org/CogVideo...,460,False,https://zai-org-cogvideox-2b-space.hf.space/api,"<!doctype html>\n<html\n\tlang=""en""\n\tstyle=""..."
2,wan2-video-generation,https://huggingface.co/spaces/OpenKing/wan2-vi...,46,False,https://openking-wan2-video-generation.hf.spac...,"<!doctype html>\n<html\n\tlang=""en""\n\tstyle=""..."
3,LTX-2.3-turbo,https://huggingface.co/spaces/ZeroCollabs/LTX-...,41,False,,
4,CogVideoX,https://huggingface.co/spaces/VideoSys/CogVideoX,27,False,,
5,HunyuanVideo,https://huggingface.co/spaces/RageshAntony/Hun...,22,False,,
6,studiomi300,https://huggingface.co/spaces/lablab-ai-amd-de...,22,False,https://lablab-ai-amd-developer-hackathon-stud...,"<!doctype html>\n<html\n\tlang=""en""\n\tstyle=""..."
7,wan2.2,https://huggingface.co/spaces/wavespeed/wan2.2,19,False,,
8,Omni-videos-custom,https://huggingface.co/spaces/CumaSystem/Omni-...,18,False,https://cumasystem-omni-videos-custom.hf.space...,"<!doctype html>\n<html\n\tlang=""en""\n\tstyle=""..."
9,wan2-video-generation-nsfw,https://huggingface.co/spaces/samsammysammy/wa...,13,False,https://samsammysammy-wan2-video-generation-ns...,"<!doctype html>\n<html\n\tlang=""en""\n\tstyle=""..."


In [2]:
from google.colab import files
files.download("hf_video_generation_spaces.csv")
files.download("hf_video_generation_spaces.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
"""
HuggingFace Video-Generation Spaces Scraper
Does exactly what your Automa flow described:

1. Goes to https://huggingface.co/spaces?category=video-generation (via API + semantic search)
2. Gets all space URLs (handles pagination)
3. For each space: extracts name = last part of URL
4. Checks for "Use via API" button
5. If present, copies the whole API page content
6. Saves everything to CSV + Excel you can download / open in Google Sheets

Install: pip install huggingface_hub requests beautifulsoup4 pandas openpyxl tqdm
Run: python hf_video_spaces_scraper.py
"""

import time
import re
import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm import tqdm
from huggingface_hub import list_spaces

# CONFIG
CATEGORY = "video-generation"
OUTPUT_CSV = "/mnt/data/hf_video_generation_spaces.csv"
OUTPUT_EXCEL = "/mnt/data/hf_video_generation_spaces.xlsx"
DELAY = 1.2  # seconds between requests to avoid rate limit
LIMIT = 500  # how many spaces to scrape, increase if you want all

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

def get_api_page_content(space_id):
    """
    Tries to replicate clicking "Use via API"
    space_id = like "observantdistressed/wan2-2-i2v-v3"
    """
    result = {
        "has_api_button": False,
        "api_docs_url": "",
        "api_docs_content": "",
        "api_error": ""
    }

    # 1. Fetch main space page
    space_url = f"https://huggingface.co/spaces/{space_id}"
    try:
        resp = requests.get(space_url, headers=HEADERS, timeout=15)
        if resp.status_code != 200:
            result["api_error"] = f"Main page status {resp.status_code}"
            return result

        soup = BeautifulSoup(resp.text, 'html.parser')
        text = resp.text

        # Check for "Use via API" presence - HF renders it in various ways
        if "Use via API" in text or "use via api" in text.lower() or "gradio_api" in text.lower() or 'View API' in text:
            result["has_api_button"] = True

        # If has API, try to get the actual API docs
        # Method A: HF hosted api page
        api_urls_to_try = [
            f"https://huggingface.co/spaces/{space_id}/api",
            f"https://huggingface.co/spaces/{space_id}/tree/main",
            f"https://{space_id.replace('/', '-').replace('_', '-').lower()}.hf.space",
        ]

        # Gradio's openapi is the real "copy whole page" target
        gradio_subdomain = space_id.replace("/", "-").replace("_", "-").lower()
        gradio_urls = [
            f"https://{gradio_subdomain}.hf.space/gradio_api/openapi",
            f"https://{gradio_subdomain}.hf.space/config",
            f"https://{gradio_subdomain}.hf.space/info",
            f"https://{gradio_subdomain}.hf.space/api",
        ]
        api_urls_to_try.extend(gradio_urls)

        for api_url in api_urls_to_try:
            try:
                r = requests.get(api_url, headers=HEADERS, timeout=12)
                if r.status_code == 200 and len(r.text) > 200:
                    result["api_docs_url"] = api_url
                    # truncate to avoid huge files, but save enough
                    result["api_docs_content"] = r.text[:30000]  # first 30k chars
                    break
            except:
                continue

        # If still empty but button exists, save the main page's API snippet
        if result["has_api_button"] and not result["api_docs_content"]:
            # extract the part that contains API info from main page
            result["api_docs_url"] = space_url
            result["api_docs_content"] = text[:30000]

    except Exception as e:
        result["api_error"] = str(e)

    return result


def main():
    print(f"Listing spaces for category: {CATEGORY}")
    # list_spaces with filter does the same as ?category=video-generation
    # According to HF docs: filter can be task. video-generation is a task.
    spaces = []
    try:
        # Try direct filter first
        for space in list_spaces(filter=CATEGORY, limit=LIMIT, sort="likes"):
            spaces.append(space)
    except Exception as e:
        print(f"filter={CATEGORY} failed: {e}, trying search")
        # fallback: semantic search
        from huggingface_hub import search_spaces
        for space in search_spaces(CATEGORY, limit=LIMIT):
            spaces.append(space)

    print(f"Found {len(spaces)} spaces. Starting detailed scrape...")

    rows = []
    for s in tqdm(spaces):
        space_id = s.id  # e.g. "observantdistressed/wan2-2-i2v-v3"
        space_url = f"https://huggingface.co/spaces/{space_id}"
        space_name = space_id.split("/")[-1]  # this is your step 3: wan2-2-i2v-v3

        api_info = get_api_page_content(space_id)

        rows.append({
            "space_id": space_id,
            "name": space_name,  # step 3
            "url": space_url,  # step 2
            "likes": getattr(s, 'likes', 0),
            "author": getattr(s, 'author', ''),
            "sdk": getattr(s, 'sdk', ''),
            "has_use_via_api_button": api_info["has_api_button"],  # step 4
            "api_docs_url": api_info["api_docs_url"],  # step 5 - where we clicked
            "api_docs_content": api_info["api_docs_content"],  # step 5 - copied page
            "error": api_info["api_error"]
        })

        time.sleep(DELAY)

    df = pd.DataFrame(rows)
    df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
    df.to_excel(OUTPUT_EXCEL, index=False)

    print(f"\nDONE! Saved {len(df)} rows")
    print(f"CSV: {OUTPUT_CSV}")
    print(f"Excel: {OUTPUT_EXCEL}")
    print(f"\nPreview:\n{df[['name','url','has_use_via_api_button']].head(10)}")

if __name__ == "__main__":
    main()


Listing spaces for category: video-generation
Found 354 spaces. Starting detailed scrape...


  5%|▍         | 17/354 [00:30<10:06,  1.80s/it]


KeyboardInterrupt: 

In [5]:
"""
Scraper for https://huggingface.co/spaces?q=Wan2.1&category=image-generation
Exact same logic as before, but filtered for Wan2.1 + image-generation

Does:
1. Search spaces with query=Wan2.1 and filter=image-generation
2. For each: gets name (last part of URL), full URL
3. Checks for "Use via API" button
4. If exists, copies API docs page
5. Saves to CSV/Excel

Run in Google Colab for full results (my server has no internet)

pip install huggingface_hub requests beautifulsoup4 pandas openpyxl tqdm
python hf_wan21_image_generation_scraper.py
"""

import time
import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm import tqdm

# Try to import huggingface_hub, if not available fallback to direct API calls
try:
    from huggingface_hub import list_spaces
    HF_HUB_AVAILABLE = True
except:
    HF_HUB_AVAILABLE = False

CATEGORY = "image-generation"
QUERY = "Wan2.1"
LIMIT = 1000
DELAY = 0.8

OUTPUT_CSV = "/mnt/data/hf_wan21_image_generation.csv"
OUTPUT_EXCEL = "/mnt/data/hf_wan21_image_generation.xlsx"

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

def get_spaces_via_api():
    """Get spaces using HF public API - works even without huggingface_hub"""
    # This endpoint mirrors what the website uses for ?q=Wan2.1&category=image-generation
    # https://huggingface.co/api/spaces?search=Wan2.1&filter=image-generation
    spaces = []
    url = f"https://huggingface.co/api/spaces?search={QUERY}&filter={CATEGORY}&limit={LIMIT}&sort=likes&direction=-1"
    print(f"Fetching: {url}")
    try:
        r = requests.get(url, headers=HEADERS, timeout=20)
        if r.status_code == 200:
            data = r.json()
            # data is list of space dicts
            for item in data:
                # normalize to object-like
                spaces.append(item)
            print(f"Got {len(spaces)} via direct API")
            return spaces
    except Exception as e:
        print(f"Direct API failed: {e}")
    return spaces

def get_spaces_via_hub():
    spaces = []
    print(f"Listing via huggingface_hub: filter={CATEGORY}, search={QUERY}")
    try:
        # huggingface_hub supports search and filter together
        for s in list_spaces(filter=CATEGORY, search=QUERY, limit=LIMIT, sort="likes"):
            spaces.append(s)
    except Exception as e:
        print(f"Hub method failed: {e}")
    return spaces

def get_api_page_content(space_id):
    result = {"has_api_button": False, "api_docs_url": "", "api_docs_content": "", "error": ""}
    space_url = f"https://huggingface.co/spaces/{space_id}"
    try:
        resp = requests.get(space_url, headers=HEADERS, timeout=15)
        text = resp.text
        if "Use via API" in text or "gradio_api" in text.lower():
            result["has_api_button"] = True

        if result["has_api_button"]:
            subdomain = space_id.replace("/", "-").replace("_", "-").lower()
            urls = [
                f"https://{subdomain}.hf.space/gradio_api/openapi",
                f"https://{subdomain}.hf.space/api",
                f"https://huggingface.co/spaces/{space_id}/api",
            ]
            for api_u in urls:
                try:
                    r = requests.get(api_u, headers=HEADERS, timeout=10)
                    if r.status_code == 200 and len(r.text) > 200:
                        result["api_docs_url"] = api_u
                        result["api_docs_content"] = r.text[:25000]
                        break
                except:
                    continue
            if not result["api_docs_content"]:
                result["api_docs_url"] = space_url
                result["api_docs_content"] = text[:25000]
    except Exception as e:
        result["error"] = str(e)
    return result

def main():
    all_spaces = []

    if HF_HUB_AVAILABLE:
        all_spaces = get_spaces_via_hub()

    if not all_spaces:
        all_spaces = get_spaces_via_api()

    if not all_spaces:
        print("No spaces found via API - you need internet. Creating empty template.")
        all_spaces = []

    print(f"Total spaces found: {len(all_spaces)}")
    rows = []

    for item in tqdm(all_spaces):
        # Handle both dict (from direct API) and object (from hub)
        if isinstance(item, dict):
            space_id = item.get("id") or item.get("_id") or ""
            likes = item.get("likes", 0)
            author = item.get("author", "")
            sdk = item.get("sdk", "")
        else:
            space_id = getattr(item, 'id', '')
            likes = getattr(item, 'likes', 0)
            author = getattr(item, 'author', '')
            sdk = getattr(item, 'sdk', '')

        if not space_id:
            continue

        name = space_id.split("/")[-1]
        url = f"https://huggingface.co/spaces/{space_id}"

        api_info = get_api_page_content(space_id)

        rows.append({
            "space_id": space_id,
            "name": name,
            "url": url,
            "likes": likes,
            "author": author,
            "sdk": sdk,
            "query_match": QUERY,
            "category": CATEGORY,
            "has_use_via_api_button": api_info["has_api_button"],
            "api_docs_url": api_info["api_docs_url"],
            "api_docs_content": api_info["api_docs_content"],
            "error": api_info["error"]
        })
        time.sleep(DELAY)

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(by="likes", ascending=False)

    df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
    df.to_excel(OUTPUT_EXCEL, index=False)
    print(f"\nSaved to {OUTPUT_CSV} - {len(df)} rows")
    print(df.head().to_string() if not df.empty else "Empty - run in Colab with internet")

if __name__ == "__main__":
    main()


Listing via huggingface_hub: filter=image-generation, search=Wan2.1
Fetching: https://huggingface.co/api/spaces?search=Wan2.1&filter=image-generation&limit=1000&sort=likes&direction=-1
Got 0 via direct API
No spaces found via API - you need internet. Creating empty template.
Total spaces found: 0


0it [00:00, ?it/s]


OSError: Cannot save file into a non-existent directory: '/mnt/data'

In [6]:
!pip install huggingface_hub pandas openpyxl tqdm -q

from huggingface_hub import list_spaces
import pandas as pd
import time, requests
from tqdm import tqdm

QUERY = "Wan2.1"
CATEGORY = "image-generation"
LIMIT = 500

HEADERS = {"User-Agent": "Mozilla/5.0"}

print(f"Searching for {QUERY} in {CATEGORY}...")

# Method 1: search only - filter manually later
spaces = list(list_spaces(search=QUERY, limit=LIMIT, sort="likes"))
print(f"Found {len(spaces)} spaces with '{QUERY}'")

# Filter for image-generation if possible
filtered = []
for s in spaces:
    tags = getattr(s, 'tags', [])
    # tags is list, check if image-generation related
    if any(CATEGORY in str(t).lower() or "image" in str(t).lower() or "wan" in str(t).lower() for t in tags) or True: # keep all Wan2.1 for now
        filtered.append(s)

print(f"After filtering: {len(filtered)} spaces")

rows = []
for s in tqdm(filtered):
    space_id = s.id
    space_url = f"https://huggingface.co/spaces/{space_id}"
    name = space_id.split("/")[-1]

    # Check for Use via API button
    has_api = False
    api_url = ""
    api_content = ""
    try:
        r = requests.get(space_url, headers=HEADERS, timeout=10)
        if "Use via API" in r.text:
            has_api = True
            # Try gradio openapi
            subdomain = space_id.replace("/", "-").replace("_", "-").lower()
            api_try = f"https://{subdomain}.hf.space/gradio_api/openapi"
            try:
                ra = requests.get(api_try, headers=HEADERS, timeout=8)
                if ra.status_code == 200:
                    api_url = api_try
                    api_content = ra.text[:20000]
                else:
                    api_url = space_url
                    api_content = r.text[:20000]
            except:
                api_url = space_url
                api_content = r.text[:15000]
    except:
        pass

    rows.append({
        "space_id": space_id,
        "name": name,  # আপনার Step 3
        "url": space_url,  # আপনার Step 2
        "likes": getattr(s, 'likes', 0),
        "author": getattr(s, 'author', ''),
        "sdk": getattr(s, 'sdk', ''),
        "has_use_via_api_button": has_api,  # Step 4
        "api_docs_url": api_url,  # Step 5
        "api_docs_content": api_content,  # Step 5 - পুরো পেজ
    })
    time.sleep(0.5)

df = pd.DataFrame(rows)
df = df.sort_values(by="likes", ascending=False)

# FIX: Colab এ /mnt/data নেই, তাই এখানে সেভ করুন
df.to_csv("hf_wan21_image_generation.csv", index=False)
df.to_excel("hf_wan21_image_generation.xlsx", index=False)

print(f"\nDONE! Total {len(df)} rows")
print(df[["name","likes","has_use_via_api_button"]].head(10).to_string())

# Download
from google.colab import files
files.download("hf_wan21_image_generation.csv")
files.download("hf_wan21_image_generation.xlsx")

Searching for Wan2.1 in image-generation...
Found 500 spaces with 'Wan2.1'
After filtering: 500 spaces


100%|██████████| 500/500 [05:00<00:00,  1.66it/s]


DONE! Total 500 rows
                    name  likes  has_use_via_api_button
0                 Wan2.1   2178                   False
1    Wan2.1-Fun-1.3B-InP     85                   False
2       Wan2.1-VACE-1.3B     72                   False
3                 Wan2.1     47                   False
4          Wan2.1-Kerala     32                   False
5       Wan2.1-DUPLICATE     18                   False
6  Wan-2.1-T2V-1.3B-LoRA      9                   False
7          Wan2.1-sajjad      5                   False
8  Wan-AI-Wan2.1-T2V-14B      5                   False
9           Wan2.1-Image      5                   False


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
!pip install gradio_client huggingface_hub pandas openpyxl -q

from gradio_client import Client
from huggingface_hub import list_spaces
import pandas as pd

def get_api_info_proper(space_id):
    """
    This is the CORRECT way to get "Use via API" content
    """
    result = {
        "has_use_via_api_button": False,
        "api_docs_url": "",
        "api_docs_content": "",
    }

    try:
        # This directly talks to the Gradio API server
        # e.g. for Wan-AI/Wan2.1-T2V-1.3B it connects to Wan-AI-Wan2.1-T2V-1.3B.hf.space
        client = Client(space_id, verbose=False)

        # If Client() succeeds, it means "Use via API" exists
        result["has_use_via_api_button"] = True
        result["api_docs_url"] = f"https://huggingface.co/spaces/{space_id}"

        # This gets the full API spec - same as copying the whole API page
        api_dict = client.view_api()
        result["api_docs_content"] = str(api_dict)[:30000]  # full spec

        print(f"✅ {space_id} has API")

    except Exception as e:
        # If it fails, it usually means:
        # 1. It's a Docker space (no API)
        # 2. Space is sleeping/paused
        # 3. Not a Gradio space
        error_msg = str(e).lower()
        if "not a valid" in error_msg or "could not get config" in error_msg:
            result["has_use_via_api_button"] = False
        else:
            # Even if sleeping, try to get config via direct HTTP
            result["has_use_via_api_button"] = False
            result["api_docs_content"] = f"Error: {e}"

    return result

# --- TEST IT FOR Wan2.1 ---
QUERY = "Wan2.1"
LIMIT = 50  # test with 50 first

spaces = list(list_spaces(search=QUERY, limit=LIMIT, sort="likes"))
print(f"Found {len(spaces)} spaces")

rows = []
for s in spaces:
    space_id = s.id
    print(f"Checking {space_id}...")
    api_info = get_api_info_proper(space_id)

    rows.append({
        "name": space_id.split("/")[-1],
        "url": f"https://huggingface.co/spaces/{space_id}",
        "likes": s.likes,
        "has_use_via_api_button": api_info["has_use_via_api_button"],
        "api_docs_url": api_info["api_docs_url"],
        "api_docs_content": api_info["api_docs_content"],
    })

df = pd.DataFrame(rows)
df.to_csv("wan21_with_api_fixed.csv", index=False)
print(df.head())

from google.colab import files
files.download("wan21_with_api_fixed.csv")

Found 50 spaces
Checking Wan-AI/Wan2.1...
Client.predict() Usage Info
---------------------------
Named API endpoints: 7

 - predict(api_name="/online_process_change") -> generated_video
    Parameters:
     - None
    Returns:
     - [Video] generated_video: dict(video: filepath, subtitles: filepath | None) 

 - predict(api_name="/switch_i2v_tab") -> 
    Parameters:
     - None
    Returns:
     - None

 - predict(api_name="/switch_t2v_tab") -> 
    Parameters:
     - None
    Returns:
     - None

 - predict(prompt, size, watermark_wan, seed, api_name="/t2v_generation_async") -> (cost_timesecs, estimated_waiting_timesecs)
    Parameters:
     - [Textbox] prompt: str (required)  
     - [Dropdown] size: Literal['1280*720', '960*960', '720*1280', '1088*832', '832*1088'] (not required, defaults to:   1280*720)  
     - [Checkbox] watermark_wan: bool (not required, defaults to:   True)  
     - [Number] seed: float (not required, defaults to:   -1)  
    Returns:
     - [Number] cost_ti

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
"""
FULL FINAL SCRAPER for https://huggingface.co/spaces?q=Wan2.1&category=image-generation
Fixed: Properly detects and copies "Use via API" page

What it does (your 5 steps):
1. Goes to https://huggingface.co/spaces?q=Wan2.1&category=image-generation
2. Gets all space URLs (with pagination)
3. Takes name = last part of URL (wan2-2-i2v-v3 etc)
4. Checks for "Use via API" button PROPERLY via Gradio Client
5. Copies whole API page content

Run in Google Colab:
!pip install huggingface_hub gradio_client pandas openpyxl tqdm requests -q
!python hf_wan21_full_final.py
"""

import time
import requests
import pandas as pd
from tqdm import tqdm

# Try to import, will work in Colab after pip install
try:
    from huggingface_hub import list_spaces
    from gradio_client import Client
    HF_AVAILABLE = True
except ImportError:
    HF_AVAILABLE = False
    print("Please run: pip install huggingface_hub gradio_client pandas openpyxl tqdm requests")

# ========= CONFIG =========
QUERY = "Wan2.1"
CATEGORY = "image-generation"  # for reference, we search by QUERY and filter manually
LIMIT = 1000  # change to 2000 for all
OUTPUT_CSV = "hf_wan21_image_generation_FULL.csv"
OUTPUT_EXCEL = "hf_wan21_image_generation_FULL.xlsx"
DELAY = 0.5

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

def get_api_info_proper(space_id):
    """
    CORRECT way to replicate clicking "Use via API"
    If Gradio Client connects, button exists and we get full API docs
    """
    result = {
        "has_use_via_api_button": False,
        "api_docs_url": "",
        "api_docs_content": "",
        "api_endpoints": "",
        "error": ""
    }

    # Try Method 1: Gradio Client (most accurate)
    try:
        # timeout=10 to avoid hanging on sleeping spaces
        client = Client(space_id, verbose=False)
        view = client.view_api()

        result["has_use_via_api_button"] = True
        result["api_docs_url"] = f"https://huggingface.co/spaces/{space_id}"
        result["api_docs_content"] = str(view)[:40000]

        # Also get readable endpoints
        try:
            endpoints = []
            for fn in view.get('named_endpoints', {}):
                endpoints.append(fn)
            result["api_endpoints"] = ", ".join(endpoints)[:5000]
        except:
            result["api_endpoints"] = "Gradio API found"

        return result
    except Exception as e:
        err = str(e).lower()
        # Common errors that mean NO API button
        if "not a valid gradio" in err or "could not get config" in err or "404" in err:
            result["has_use_via_api_button"] = False
            result["error"] = f"No API: {str(e)[:200]}"
            return result
        # Else try Method 2: Direct HTTP fallback (for sleeping spaces)
        try:
            subdomain = space_id.replace("/", "-").replace("_", "-").lower()
            urls_to_try = [
                f"https://{subdomain}.hf.space/gradio_api/openapi",
                f"https://{subdomain}.hf.space/config",
                f"https://huggingface.co/spaces/{space_id}",
            ]
            for api_url in urls_to_try:
                try:
                    r = requests.get(api_url, headers=HEADERS, timeout=10)
                    if r.status_code == 200 and len(r.text) > 500:
                        # Check if it contains API info
                        if "openapi" in r.text.lower() or "gradio" in r.text.lower() or "Use via API" in r.text:
                            result["has_use_via_api_button"] = True
                            result["api_docs_url"] = api_url
                            result["api_docs_content"] = r.text[:40000]
                            return result
                except:
                    continue
        except Exception as e2:
            result["error"] = str(e2)[:200]

    return result

def main():
    print(f"=== Scraping HuggingFace Spaces ===")
    print(f"Query: {QUERY} | Category: {CATEGORY} | Limit: {LIMIT}")

    if not HF_AVAILABLE:
        print("ERROR: huggingface_hub not installed. Run pip install first.")
        return

    print(f"\n[Step 1] Listing spaces from https://huggingface.co/spaces?q={QUERY}&category={CATEGORY}")
    spaces = list(list_spaces(search=QUERY, limit=LIMIT, sort="likes"))
    print(f"Found {len(spaces)} spaces matching '{QUERY}'")

    # Optional: filter for image-generation tag (keep all Wan2.1 if you want everything)
    # filtered_spaces = [s for s in spaces if any('image' in str(t).lower() for t in getattr(s, 'tags', []))]
    # print(f"After image-generation filter: {len(filtered_spaces)}")
    filtered_spaces = spaces  # keep all Wan2.1

    rows = []
    print(f"\n[Step 2-5] Checking each space for Use via API button...")

    for s in tqdm(filtered_spaces, desc="Processing spaces"):
        space_id = s.id
        name = space_id.split("/")[-1]  # Step 3
        url = f"https://huggingface.co/spaces/{space_id}"  # Step 2

        api_info = get_api_info_proper(space_id)  # Step 4 & 5

        rows.append({
            "space_id": space_id,
            "name": name,
            "url": url,
            "likes": getattr(s, 'likes', 0),
            "author": getattr(s, 'author', ''),
            "sdk": getattr(s, 'sdk', ''),
            "tags": ", ".join(getattr(s, 'tags', [])[:10]),
            "has_use_via_api_button": api_info["has_use_via_api_button"],
            "api_docs_url": api_info["api_docs_url"],
            "api_endpoints": api_info["api_endpoints"],
            "api_docs_content": api_info["api_docs_content"],
            "error": api_info["error"],
        })

        time.sleep(DELAY)

    # Save
    df = pd.DataFrame(rows)
    df = df.sort_values(by="likes", ascending=False)

    df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
    df.to_excel(OUTPUT_EXCEL, index=False)

    print(f"\n{'='*60}")
    print(f"DONE! Saved {len(df)} spaces")
    print(f"CSV: {OUTPUT_CSV}")
    print(f"Excel: {OUTPUT_EXCEL}")
    print(f"{'='*60}")
    print(f"\nWith API button: {df['has_use_via_api_button'].sum()} / {len(df)}")
    print("\nTop 10:")
    print(df[["name", "likes", "has_use_via_api_button"]].head(10).to_string(index=False))

if __name__ == "__main__":
    main()


=== Scraping HuggingFace Spaces ===
Query: Wan2.1 | Category: image-generation | Limit: 1000

[Step 1] Listing spaces from https://huggingface.co/spaces?q=Wan2.1&category=image-generation
Found 715 spaces matching 'Wan2.1'

[Step 2-5] Checking each space for Use via API button...


Processing spaces:   0%|          | 0/715 [00:00<?, ?it/s]

Client.predict() Usage Info
---------------------------
Named API endpoints: 7

 - predict(api_name="/online_process_change") -> generated_video
    Parameters:
     - None
    Returns:
     - [Video] generated_video: dict(video: filepath, subtitles: filepath | None) 

 - predict(api_name="/switch_i2v_tab") -> 
    Parameters:
     - None
    Returns:
     - None

 - predict(api_name="/switch_t2v_tab") -> 
    Parameters:
     - None
    Returns:
     - None

 - predict(prompt, size, watermark_wan, seed, api_name="/t2v_generation_async") -> (cost_timesecs, estimated_waiting_timesecs)
    Parameters:
     - [Textbox] prompt: str (required)  
     - [Dropdown] size: Literal['1280*720', '960*960', '720*1280', '1088*832', '832*1088'] (not required, defaults to:   1280*720)  
     - [Checkbox] watermark_wan: bool (not required, defaults to:   True)  
     - [Number] seed: float (not required, defaults to:   -1)  
    Returns:
     - [Number] cost_timesecs: float 
     - [Number] estimated_w

Processing spaces:   0%|          | 1/715 [00:00<11:31,  1.03it/s]

Client.predict() Usage Info
---------------------------
Named API endpoints: 6

 - predict(api_name="/select_template") -> (the_image_at_the_beginning_of_the_video_图片到视频的开始图片, prompt_正向提示词)
    Parameters:
     - None
    Returns:
     - [Image] the_image_at_the_beginning_of_the_video_图片到视频的开始图片: dict(path: str | None (Path to a local file), url: str | None (Publicly available url or base64 encoded image), size: int | None (Size of image in bytes), orig_name: str | None (Original filename), mime_type: str | None (mime type of image), is_stream: bool (Can always be set to False), meta: dict()) 
     - [Textbox] prompt_正向提示词: str 

 - predict(api_name="/lambda") -> seed_随机种子
    Parameters:
     - None
    Returns:
     - [Textbox] seed_随机种子: str 

 - predict(generation_method, api_name="/upload_generation_method") -> animation_length_视频帧数
    Parameters:
     - [Radio] generation_method: Literal['Video Generation', 'Image Generation'] (not required, defaults to:   Video Generation)  
  

...

Client.predict() Usage Info
---------------------------
Named API endpoints: 1

 - predict(prompt, negative_prompt, seed, randomize_seed, width, height, guidance_scale, num_inference_steps, api_name="/infer") -> (result, seed)
    Parameters:
     - [Textbox] prompt: str (required)  
     - [Textbox] negative_prompt: str (required)  
     - [Slider] seed: float (not required, defaults to:   0)  
     - [Checkbox] randomize_seed: bool (not required, defaults to:   True)  
     - [Slider] width: float (not required, defaults to:   1024)  
     - [Slider] height: float (not required, defaults to:   1024)  
     - [Slider] guidance_scale: float (not required, defaults to:   0.0)  
     - [Slider] num_inference_steps: float (not required, defaults to:   2)  
    Returns:
     - [Image] result: dict(path: str | None (Path to a local file), url: str | None (Publicly available url or base64 encoded image), size: int | None (Size of image in bytes), orig_name: str | None (Original filename), mi

Processing spaces:  98%|█████████▊| 701/715 [11:34<00:11,  1.17it/s]

Client.predict() Usage Info
---------------------------
Named API endpoints: 1

 - predict(prompt, first_frame, last_frame, ref1, ref2, ref3, num_frames, num_steps, guidance, seed, randomize_seed, api_name="/run") -> result
    Parameters:
     - [Textbox] prompt: str (required)  
     - [Image] first_frame: filepath (required)  (A local filepath or publicly available URL.) 
     - [Image] last_frame: filepath (required)  (A local filepath or publicly available URL.) 
     - [Image] ref1: filepath (required)  (A local filepath or publicly available URL.) 
     - [Image] ref2: filepath (required)  (A local filepath or publicly available URL.) 
     - [Image] ref3: filepath (required)  (A local filepath or publicly available URL.) 
     - [Slider] num_frames: float (not required, defaults to:   81)  
     - [Slider] num_steps: float (not required, defaults to:   30)  
     - [Slider] guidance: float (not required, defaults to:   5.0)  
     - [Slider] seed: float (not required, defaults 

Processing spaces: 100%|██████████| 715/715 [11:46<00:00,  1.01it/s]



DONE! Saved 715 spaces
CSV: hf_wan21_image_generation_FULL.csv
Excel: hf_wan21_image_generation_FULL.xlsx

With API button: 321 / 715

Top 10:
                 name  likes  has_use_via_api_button
               Wan2.1   2178                    True
  Wan2.1-Fun-1.3B-InP     85                    True
     Wan2.1-VACE-1.3B     72                    True
               Wan2.1     47                    True
        Wan2.1-Kerala     32                   False
     Wan2.1-DUPLICATE     18                    True
Wan-2.1-T2V-1.3B-LoRA      9                    True
        Wan2.1-sajjad      5                   False
Wan-AI-Wan2.1-T2V-14B      5                    True
         Wan2.1-Image      5                    True


In [9]:
!pip install huggingface_hub gradio_client pandas openpyxl tqdm requests -q

import time, requests, pandas as pd
from tqdm import tqdm
from huggingface_hub import list_spaces
from gradio_client import Client

QUERY = "Wan2.1"
LIMIT = 500
HEADERS = {"User-Agent": "Mozilla/5.0"}

def get_api_info_proper(space_id):
    result = {"has_use_via_api_button": False, "api_docs_url": "", "api_docs_content": "", "api_endpoints": "", "error": ""}
    try:
        client = Client(space_id, verbose=False)
        view = client.view_api()
        result["has_use_via_api_button"] = True
        result["api_docs_url"] = f"https://huggingface.co/spaces/{space_id}"
        result["api_docs_content"] = str(view)[:40000]
        return result
    except Exception as e:
        # Fallback for sleeping spaces
        try:
            subdomain = space_id.replace("/", "-").lower()
            r = requests.get(f"https://{subdomain}.hf.space/gradio_api/openapi", headers=HEADERS, timeout=8)
            if r.status_code == 200:
                result["has_use_via_api_button"] = True
                result["api_docs_url"] = f"https://{subdomain}.hf.space/gradio_api/openapi"
                result["api_docs_content"] = r.text[:40000]
                return result
        except:
            pass
        result["error"] = str(e)[:200]
        return result

spaces = list(list_spaces(search=QUERY, limit=LIMIT, sort="likes"))
print(f"Found {len(spaces)} spaces")

rows = []
for s in tqdm(spaces):
    space_id = s.id
    api_info = get_api_info_proper(space_id)
    rows.append({
        "space_id": space_id,
        "name": space_id.split("/")[-1],
        "url": f"https://huggingface.co/spaces/{space_id}",
        "likes": s.likes,
        "has_use_via_api_button": api_info["has_use_via_api_button"],
        "api_docs_url": api_info["api_docs_url"],
        "api_docs_content": api_info["api_docs_content"],
    })
    time.sleep(0.5)

df = pd.DataFrame(rows).sort_values(by="likes", ascending=False)
df.to_csv("hf_wan21_image_generation_FULL.csv", index=False)
df.to_excel("hf_wan21_image_generation_FULL.xlsx", index=False)

print(f"DONE! With API: {df['has_use_via_api_button'].sum()} / {len(df)}")
print(df.head(10)[["name","likes","has_use_via_api_button"]])

from google.colab import files
files.download("hf_wan21_image_generation_FULL.csv")

Found 500 spaces


  0%|          | 0/500 [00:00<?, ?it/s]

Client.predict() Usage Info
---------------------------
Named API endpoints: 7

 - predict(api_name="/online_process_change") -> generated_video
    Parameters:
     - None
    Returns:
     - [Video] generated_video: dict(video: filepath, subtitles: filepath | None) 

 - predict(api_name="/switch_i2v_tab") -> 
    Parameters:
     - None
    Returns:
     - None

 - predict(api_name="/switch_t2v_tab") -> 
    Parameters:
     - None
    Returns:
     - None

 - predict(prompt, size, watermark_wan, seed, api_name="/t2v_generation_async") -> (cost_timesecs, estimated_waiting_timesecs)
    Parameters:
     - [Textbox] prompt: str (required)  
     - [Dropdown] size: Literal['1280*720', '960*960', '720*1280', '1088*832', '832*1088'] (not required, defaults to:   1280*720)  
     - [Checkbox] watermark_wan: bool (not required, defaults to:   True)  
     - [Number] seed: float (not required, defaults to:   -1)  
    Returns:
     - [Number] cost_timesecs: float 
     - [Number] estimated_w

  0%|          | 1/500 [00:00<07:59,  1.04it/s]

Client.predict() Usage Info
---------------------------
Named API endpoints: 6

 - predict(api_name="/select_template") -> (the_image_at_the_beginning_of_the_video_图片到视频的开始图片, prompt_正向提示词)
    Parameters:
     - None
    Returns:
     - [Image] the_image_at_the_beginning_of_the_video_图片到视频的开始图片: dict(path: str | None (Path to a local file), url: str | None (Publicly available url or base64 encoded image), size: int | None (Size of image in bytes), orig_name: str | None (Original filename), mime_type: str | None (mime type of image), is_stream: bool (Can always be set to False), meta: dict()) 
     - [Textbox] prompt_正向提示词: str 

 - predict(api_name="/lambda") -> seed_随机种子
    Parameters:
     - None
    Returns:
     - [Textbox] seed_随机种子: str 

 - predict(generation_method, api_name="/upload_generation_method") -> animation_length_视频帧数
    Parameters:
     - [Radio] generation_method: Literal['Video Generation', 'Image Generation'] (not required, defaults to:   Video Generation)  
  

...

Client.predict() Usage Info
---------------------------
Named API endpoints: 1

 - predict(api_name="/_check_login_status") -> 
    Parameters:
     - None
    Returns:
     - None



100%|██████████| 500/500 [07:26<00:00,  1.12it/s]

DONE! With API: 28 / 500
                    name  likes  has_use_via_api_button
0                 Wan2.1   2178                    True
1    Wan2.1-Fun-1.3B-InP     85                    True
2       Wan2.1-VACE-1.3B     72                    True
3                 Wan2.1     47                    True
4          Wan2.1-Kerala     32                   False
5       Wan2.1-DUPLICATE     18                   False
6  Wan-2.1-T2V-1.3B-LoRA      9                   False
7          Wan2.1-sajjad      5                   False
8  Wan-AI-Wan2.1-T2V-14B      5                    True
9           Wan2.1-Image      5                   False


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>